# 51% attack analysis: does the long-private-chain defense work?

Four questions, answered from simulation CSVs produced by `SimulationRunner` and `FalsePositiveExperiment`:

1. How does the double-spend success rate change with attacker hash share, with and without the defense?
2. What does the final chain look like (orphan rate, who owns the blocks)?
3. What does the defense cost in runtime and memory?
4. Are large honest miners (30% to 45%) wrongly flagged?

The defense is `LongPrivateChainRule` (limit 6) with Safe Mode: a block that makes six in a row from one miner is checked against the main chain, and rejected only if its chain re-spends coins differently (fraud). Otherwise it is accepted and counted as a false positive.

**Data.** The runner's main CSV has the 12 required columns. Block ownership needs three more, so this notebook uses `results/results_detailed.csv` (same columns plus block counts). Files are created on the first run and reused after that; delete `results/` to regenerate. The conclusions below describe the default run (seed 42, 100 trials per grid cell, 5 honest miners).

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

from simulation.false_positives import FalsePositiveExperiment
from simulation.plots import (
    apply_style,
    plot_chain_structure,
    plot_false_positives,
    plot_overhead,
    plot_success_vs_share,
)
from simulation.runner import SimulationRunner

apply_style()
RESULTS_DIR = Path("../results")
GRID_CSV = RESULTS_DIR / "results_detailed.csv"
FALSE_POSITIVE_CSV = RESULTS_DIR / "false_positives.csv"
SEED = 42

## Load (or generate) the data

In [ ]:
if not GRID_CSV.exists():
    SimulationRunner(trials=100, seed=SEED, detailed=True).run_to_csv(GRID_CSV)
if not FALSE_POSITIVE_CSV.exists():
    FalsePositiveExperiment(trials=100, seed=SEED).run_to_csv(FALSE_POSITIVE_CSV)

results = pd.read_csv(GRID_CSV)
false_positives = pd.read_csv(FALSE_POSITIVE_CSV)
print(f"{len(results):,} attack trials, {len(false_positives):,} false-positive trials")
results.head()

## 1. Attack success rate vs attacker hash share

In [ ]:
plot_success_vs_share(results)
plt.show()

**Conclusion.** Success climbs steeply with hash share, and past 50% the attack almost always works without the defense. The defense only helps once the attacker needs a long private chain. At 1 and 3 confirmations the attacker overtakes the honest chain within about five blocks, before the six-block rule can fire, so the gain is moderate (for example 92% down to 60% at 50% share and 3 confirmations). At 6 confirmations the attacker must build a longer chain and the defense stops every attack (83% down to 0% at 50% share).

## 2. Final chain structure: orphan rate and miner block distribution

In [ ]:
plot_chain_structure(results, threshold=6)
plt.show()

**Conclusion.** A successful reorg discards honest blocks, so the orphan rate rises to roughly 35% to 38% at 50% to 60% share. Without the defense a 50% attacker ends up owning about 88% of the final chain's blocks and a 60% attacker all of them; with the defense those drop to about 44% and 54%, because blocked attacks leave the honest chain in place. Single-miner streaks reach over 100 blocks without the defense (50% share) but stay near 20 with it.

## 3. Runtime and memory overhead of the defense

Medians per trial, pooled over confirmation depths. Memory is peak Python allocation from `tracemalloc`, measured in a separate replay of the same seeded trial so timing stays untraced.

In [ ]:
plot_overhead(results)
plt.show()
results.groupby("scenario")[["runtime_ms", "memory_mb"]].median().round(4)

**Conclusion.** The defense is cheap: median runtime rises by about 6% (4.40 ms to 4.64 ms per trial) and peak memory by about 3%. The rule only does extra work, a chain comparison, when a run of six or more blocks appears. Trials take milliseconds, so treat these as order-of-magnitude figures that vary between machines.

## 4. False positives: honest miners with 30% to 45% share

An honest-only network where one miner holds the share on the x-axis and nobody attacks, over 144 blocks. Every flagged block here is a false positive.

In [ ]:
plot_false_positives(false_positives)
plt.show()
(
    false_positives.groupby("miner_share")[["flagged", "flagged_rate"]]
    .mean()
    .rename(columns={"flagged": "runs_flagged", "flagged_rate": "blocks_flagged"})
    .round(3)
)

**Conclusion.** Large honest miners do trigger the rule: about 7% of 144-block runs contain a flagged block at 30% share and about 56% at 45%, while only 0.3% to 2% of an individual miner's blocks are flagged. Because Safe Mode accepts blocks that have no conflicting spends, none of these honest blocks is rejected. They are only logged, so a false positive costs an extra check and not lost rewards. Splitting hash power across several identities (a Sybil attacker) would avoid the six-in-a-row trigger entirely, which this defense does not address.